# Mushroom Observation Data Pipeline - Kaggle Notebook

This notebook runs the environmental data enrichment pipeline for mushroom observations from iNaturalist.

Every environmental layer is sampled from **Google Earth Engine** at the observation
points, so the run no longer downloads bulk rasters — no ERA5 netCDFs from the CDS
queue, no per-day CHIRPS GeoTIFFs, no multi-gigabyte WorldCover tiles, and no SRTM
DEM to process locally.

## Setup Instructions

### Before Running:
1. **Enable Internet** in Kaggle notebook settings (Settings → Internet → On)
2. **Add Secrets** (Kaggle Add-ons → Secrets):
   - `EARTHENGINE_PROJECT`: Your Google Cloud project ID — this is the only
     credential the pipeline needs
   - Optional, fallback only: `OPENTOPOGRAPHY_API_KEY`, `CDSAPI_URL` / `CDSAPI_KEY`.
     These are used solely when Earth Engine is unavailable, or when you set
     `FETCH_RASTERS=1` to download the source rasters on purpose.

### What comes from Earth Engine:

| Column | Dataset |
| --- | --- |
| `ndvi` | `COPERNICUS/S2_SR_HARMONIZED` |
| `soil_moisture` | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| `prcp_d0..d6` | `UCSB-CHG/CHIRPS/DAILY` |
| `tmax_d0..d6`, `tmin_d0..d6` | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| `land_cover` | `ESA/WorldCover/v200` |
| `elevation`, `slope`, `aspect` | `USGS/SRTMGL1_003` |
| `solar_exposure`, `wind_exposure`, `water_retention` | derived from the sampled terrain + `MERIT/Hydro/v1_0_1` |

### Data Storage:
Kaggle provides `/kaggle/working/` for output files that persist after the session.

## Step 1: Install Dependencies

In [ ]:
!pip install -q pyinaturalist earthengine-api requests rasterio numpy scipy pandas scikit-learn

# Fallback-only extras (bulk raster downloads, used when Earth Engine is
# unavailable or FETCH_RASTERS=1):
# !pip install -q cdsapi xarray netCDF4 rio-cogeo meteostat

In [ ]:
# if needed load the git repo
!git clone https://github.com/skyfly200/data-map.git
%cd data-map

## Step 2: Configure Environment

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Set up working directory for Kaggle
WORKING_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORKING_DIR, 'data')
os.environ['DATA_DIR'] = DATA_DIR

# Load secrets from Kaggle Add-ons
user_secrets = UserSecretsClient()

# The only credential the Earth Engine path needs.
try:
    ee_project = user_secrets.get_secret('EARTHENGINE_PROJECT')
    os.environ['EARTHENGINE_PROJECT'] = ee_project
    print("\u2713 Earth Engine project ID loaded")
except Exception as e:
    print(f"\u26a0 Earth Engine project not found: {e}")
    print("  Add it in Add-ons \u2192 Secrets \u2192 EARTHENGINE_PROJECT")
    print("  Without it the pipeline falls back to bulk raster downloads.")

# Optional, fallback only: these are read when Earth Engine is unavailable, or
# when FETCH_RASTERS=1 asks for the source rasters on purpose.
try:
    os.environ['OPENTOPOGRAPHY_API_KEY'] = user_secrets.get_secret('OPENTOPOGRAPHY_API_KEY')
    print("\u2713 OpenTopography API key loaded (raster fallback)")
except Exception:
    print("\u2139 No OpenTopography key \u2014 not needed while Earth Engine is available")

try:
    cds_url = user_secrets.get_secret('CDSAPI_URL')
    cds_key = user_secrets.get_secret('CDSAPI_KEY')
    os.environ['CDSAPI_URL'] = cds_url
    os.environ['CDSAPI_KEY'] = cds_key
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: {cds_url}\nkey: {cds_key}\n')
    print("\u2713 CDS API credentials loaded (raster fallback)")
except Exception:
    print("\u2139 No CDS credentials \u2014 not needed while Earth Engine is available")

# Create necessary directories
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'species'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'enriched'), exist_ok=True)

print(f"\nData directory: {DATA_DIR}")
print(f"Working directory: {WORKING_DIR}")

## Step 3: Initialize Earth Engine

In [ ]:
import ee

# Authenticate and initialize Earth Engine
# In Kaggle, this uses service account or OAuth based on your setup
try:
    ee.Initialize(project=os.environ.get('EARTHENGINE_PROJECT'))
    print("✓ Earth Engine initialized successfully")
except Exception as e:
    print(f"⚠ Earth Engine initialization failed: {e}")
    print("  Try running: ee.Authenticate() first, then ee.Initialize()")
    # Fallback attempt
    try:
        ee.Initialize()
        print("✓ Earth Engine initialized with default credentials")
    except Exception as e2:
        print(f"⚠ Earth Engine not available: {e2}")
        print("  NDVI and satellite moisture layers will be skipped")

## Step 4: Run the Pipeline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

# Import pipeline modules
import species_store as store
import ee_enrich
import enrich_with_rasters as enrich
import cluster
import export_geojson

print("Pipeline modules loaded successfully")
print(f"Earth Engine sampling enabled: {ee_enrich.earth_engine_enabled()}")

### 4.1: Fetch iNaturalist Observations

In [ ]:
# Run iNaturalist fetch
# This pulls mushroom observations and saves them to data/species/

print("Fetching iNaturalist observations...")
%run /kaggle/working/iNat.py

# Check what was fetched
species_files = list(store.species_slugs(store.SPECIES_DIR))
print(f"\n✓ Fetched {len(species_files)} species")
if species_files:
    print(f"  Species: {', '.join(species_files[:5])}{'...' if len(species_files) > 5 else ''}")

### 4.2: Enrich Observations from Earth Engine

In [ ]:
# Sample every environmental layer at the observation points and write
# data/enriched/<species>.csv.
#
# Earth Engine stages run first (terrain, land cover, soil moisture, rainfall,
# temperature, NDVI). The cached-raster stages then run as a fallback; each only
# touches rows still missing its column, so they cost nothing when EE succeeded.
#
# To use the old bulk-download path instead, set these before running:
#   os.environ['USE_EARTH_ENGINE'] = '0'   # or SKIP_EARTH_ENGINE=1
#   os.environ['FETCH_RASTERS'] = '1'      # then %run fetch.py and terrain_pipeline.py first

print("Enriching observations...")
%run /kaggle/working/enrich_with_rasters.py

print("\n\u2713 Enrichment complete")

### 4.3: Review Enriched Columns

In [ ]:
import pandas as pd

df = store.load_all(store.ENRICHED_DIR)
print(f"{len(df)} enriched observations, {df['species'].nunique()} species\n")

# Coverage per environmental column — how much Earth Engine actually filled in.
cols = ['ndvi', 'soil_moisture', 'land_cover', 'elevation', 'slope', 'aspect',
        'solar_exposure', 'wind_exposure', 'water_retention',
        'prcp_d0', 'prcp_d6', 'tmax_d0', 'tmin_d6']
present = [c for c in cols if c in df.columns]
coverage = pd.DataFrame({
    'filled': [df[c].notna().sum() for c in present],
    'missing': [df[c].isna().sum() for c in present],
    'coverage': [f"{df[c].notna().mean() * 100:.1f}%" for c in present],
}, index=present)
print(coverage.to_string())

### 4.4: Cluster Observations

In [ ]:
# KMeans clustering by environmental similarity
# Adds 'cluster' column to enriched CSVs

print("Clustering observations by environmental similarity...")
%run /kaggle/working/cluster.py

print("\n\u2713 Clustering complete")

### 4.5: Export GeoJSON for Visualization

In [ ]:
# Export to GeoJSON format for mapping
# Output: public/data/observations.geojson (or working directory equivalent)

print("Exporting GeoJSON...")
%run /kaggle/working/export_geojson.py

geojson_path = f"{WORKING_DIR}/observations.geojson"
if os.path.exists(geojson_path):
    size_mb = os.path.getsize(geojson_path) / (1024 * 1024)
    print(f"\n\u2713 Exported: {geojson_path} ({size_mb:.2f} MB)")
else:
    alt_path = f"{WORKING_DIR}/public/data/observations.geojson"
    if os.path.exists(alt_path):
        size_mb = os.path.getsize(alt_path) / (1024 * 1024)
        print(f"\n\u2713 Exported: {alt_path} ({size_mb:.2f} MB)")
    else:
        print("\u26a0 GeoJSON not found \u2014 check the export step's output above")

## Step 5: Review Results

In [ ]:
import pandas as pd
from pathlib import Path

# List all output files
print("=== Output Files ===")
for root, dirs, files in os.walk(WORKING_DIR):
    # Skip hidden directories and __pycache__
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace(WORKING_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit to first 10 files per directory
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        if size > 1024 * 1024:
            size_str = f"{size / (1024*1024):.1f}MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f}KB"
        else:
            size_str = f"{size}B"
        print(f'{subindent}{file} ({size_str})')
    if len(files) > 10:
        print(f'{subindent}... and {len(files) - 10} more files')

## Step 6: Download Results (Optional)

In [ ]:
# Create a zip archive of all results for download
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_name = f'mushroom_data_{timestamp}'
archive_path = shutil.make_archive(
    os.path.join(WORKING_DIR, archive_name),
    'zip',
    WORKING_DIR,
    base_dir='.'
)

print(f"✓ Created archive: {archive_path}")
print(f"  Size: {os.path.getsize(archive_path) / (1024*1024):.1f} MB")
print("\nTo download:")
print("1. Click on the 'Output' tab in Kaggle")
print(f"2. Find and download {archive_name}.zip")
print("   OR use the file browser on the right panel")

## Notes & Troubleshooting

### Common Issues:

1. **Earth Engine Authentication**: If EE initialization fails:
   ```python
   ee.Authenticate()
   ee.Initialize(project='your-project-id')
   ```
   Without a working Earth Engine session the pipeline falls back to bulk raster
   downloads, which need `OPENTOPOGRAPHY_API_KEY` and CDS credentials and take
   dramatically longer.

2. **Earth Engine quotas**: sampling is batched by observation date — one request
   per date, not per observation — but a very large date range can still hit the
   concurrent-request limit. Lower the parallelism if you see quota errors:
   ```python
   ee_enrich.enrich_precip_ee(df, max_workers=4)
   ```

3. **Memory Limits**: Kaggle notebooks have ~16GB RAM. If you hit limits:
   - Reduce the number of species being processed
   - Use `REFRESH_ALL=0` to skip re-fetching existing data
   - Process species in batches

4. **Session Timeouts**: Kaggle sessions timeout after ~12 hours. Save outputs frequently:
   ```python
   # Commit intermediate results
   %run /kaggle/working/export_geojson.py
   ```

### Resuming Interrupted Runs:

The pipeline is designed to be resumable. Every enrichment stage only samples rows
whose column is still empty, so re-running simply continues:
- Re-run the enrichment cell if it failed (it skips rows already filled)
- The `.done` marker in `data/enriched/` prevents re-processing a finished run

### Using the raster fallback deliberately:

```python
os.environ['USE_EARTH_ENGINE'] = '0'
os.environ['FETCH_RASTERS'] = '1'
%run /kaggle/working/fetch.py
%run /kaggle/working/terrain_pipeline.py
%run /kaggle/working/enrich_with_rasters.py
```

### Next Steps:

After downloading the results:
1. Upload `observations.geojson` to the Nuxt frontend's `public/data/` folder
2. Deploy to Netlify or serve statically
3. Or continue analysis in this notebook with the enriched CSVs